# Amazon Best Sellers Scraper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/amazon-bestsellers.ipynb)

**What:** Amazon Best Sellers, New Releases, Movers & Shakers, Most Wished For and Gift Ideas ranks in 21 countries: rank, ASIN, title, price, rating, reviews, images, category path. Subcategory crawl, rank-change monitor, optional product details.

**Why:** run the [`amazon-bestsellers`](https://apify.com/transparent_meteorite/amazon-bestsellers) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/amazon-bestsellers) for every option.

In [2]:
run_input = {   'categories': ['electronics'],
    'listTypes': ['bestsellers'],
    'countryCode': 'US',
    'maxItemsPerCategory': 5}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/amazon-bestsellers').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.amazon-bestsellers runId:klL8MLNS8Y4SPlckt] -> Status: RUNNING, Message: 


[apify.amazon-bestsellers runId:klL8MLNS8Y4SPlckt] -> 2026-10-14T07:15:45.607Z ACTOR: Pulling container image of build 54b9OtQUpKMqPvyI8 from registry.


[apify.amazon-bestsellers runId:klL8MLNS8Y4SPlckt] -> 2026-10-14T07:15:46.657Z ACTOR: Creating container.


[apify.amazon-bestsellers runId:klL8MLNS8Y4SPlckt] -> 2026-10-14T07:15:46.725Z ACTOR: Starting container.


[apify.amazon-bestsellers runId:klL8MLNS8Y4SPlckt] -> 2026-10-14T07:15:46.726Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.amazon-bestsellers runId:klL8MLNS8Y4SPlckt] -> 2026-10-14T07:16:07.712Z Done. Delivered 5 rows (charged 5) from 1 list(s); empty 0, failed 0, filtered 0; requests 2, ok 2, blocked 0.


[apify.amazon-bestsellers runId:klL8MLNS8Y4SPlckt] -> 2026-10-14T07:16:07.790Z INFO  [Status message]: Done: 5 items


[apify.amazon-bestsellers runId:klL8MLNS8Y4SPlckt] -> Status: RUNNING, Message: Done: 5 items


[apify.amazon-bestsellers runId:klL8MLNS8Y4SPlckt] -> 2026-10-14T07:16:07.889Z Thanks for using Amazon Best Sellers. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/z2F9Poil2f2WkMbdq/issues) with the run URL and we will reply there.


Run status: SUCCEEDED | items: 5


## 4. Tabulate

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["rank", "asin", "title", "price", "currency", "stars", "reviewsCount", "listType", "category", "countryCode"] if c in df.columns]
df[SHOW]

,rank,asin,title,price,currency,stars,reviewsCount,listType,category,countryCode
0,1,B08JHCVHTY,blink plus plan with monthly auto-renewal,11.99,USD,4.4,281067,bestsellers,Electronics,US
1,2,B0DCH8VDXF,"Apple EarPods Headphones with USB-C Plug, Wired Ear Buds with Buil...",19.00,USD,4.5,18920,bestsellers,Electronics,US
2,3,B0GJTFXNRX,"Apple AirTag (2nd Generation): Tracker for Keychain, Wallet, and M...",24.00,USD,4.6,11830,bestsellers,Electronics,US
3,4,B0FQFB8FMG,Apple AirPods Pro 3 Wireless Earbuds with Active Noise Cancellatio...,199.00,USD,4.4,16409,bestsellers,Electronics,US
4,5,B0DJGDC3BD,"Amazon Fire TV Stick HD (newest model), free & live TV, Alexa Voic...",39.99,USD,4.0,6436,bestsellers,Electronics,US


## 5. A quick analysis

In [5]:
for c in ['rank','price','stars','reviewsCount']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
print(df.groupby('listType')['price'].agg(['count','median']))
df.sort_values('rank')[['rank','title','price','stars','reviewsCount']].head(10)

             count  median
listType                  
bestsellers      5    24.0


,rank,title,price,stars,reviewsCount
0,1,blink plus plan with monthly auto-renewal,11.99,4.4,281067
1,2,"Apple EarPods Headphones with USB-C Plug, Wired Ear Buds with Buil...",19.00,4.5,18920
2,3,"Apple AirTag (2nd Generation): Tracker for Keychain, Wallet, and M...",24.00,4.6,11830
3,4,Apple AirPods Pro 3 Wireless Earbuds with Active Noise Cancellatio...,199.00,4.4,16409
4,5,"Amazon Fire TV Stick HD (newest model), free & live TV, Alexa Voic...",39.99,4.0,6436


## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [amazon-bestsellers on the Apify Store](https://apify.com/transparent_meteorite/amazon-bestsellers).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.